In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [8]:
!pip install -q \
transformers \
accelerate \
bitsandbytes \
huggingface_hub \
datasets \
sentencepiece \
safetensors

In [9]:
import numpy
import torch
import transformers

print("NumPy:", numpy.__version__)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)

NumPy: 1.26.4
Torch: 2.10.0+cu128
Transformers: 5.0.0


In [10]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [11]:
import transformers
print(transformers.__version__)

5.0.0


In [12]:
import bitsandbytes
print(bitsandbytes.__version__)

0.49.2


In [13]:
# hf token verification 
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF token")

login(token=hf_token)

print("HF Login Successful")

HF Login Successful


In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.1-8B"
)

print("Tokenizer Loaded Successfully")

Tokenizer Loaded Successfully


In [15]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.1-8B"
)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B",
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model Loaded Successfully")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model Loaded Successfully


In [16]:
#architecture verification 
print("Layers:", len(model.model.layers))
print("Hidden Size:", model.config.hidden_size)

Layers: 32
Hidden Size: 4096


In [17]:
#inference test 
prompt = "The sky was clear and the lake was calm."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


The sky was clear and the lake was calm. The weather was perfect for a day of fishing on the lake. I was excited to get out there and try my luck at catching some fish.
I had heard that the fish were biting really well lately, and I was hoping to catch a few for


In [18]:
#we will verify the gpu here 
# --- GPU diagnostics ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA compute capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    raise RuntimeError("No CUDA device found. This notebook requires a GPU.")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device: Tesla T4
CUDA compute capability: (7, 5)
Total VRAM: 14.56 GB


In [19]:
!pip install -q "numpy<2.0"

In [ ]:
import os
os._exit(0)

In [1]:
#now let's install all our dependencies
!pip install -q repeng transformers accelerate bitsandbytes

import os
import torch
import gc
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from repeng import ControlVector, ControlModel, DatasetEntry

#now configure everything here
MODEL_NAME = "meta-llama/Llama-3.1-8B"
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF token")

In [2]:
#4-bit NF4 quantazation will be done here with loading the model in 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
print("Loading Model........")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map = "auto",
    token = HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token = HF_TOKEN)

#to avoid warnings during generation now we will add EOS.

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

#architecture summary
num_layers = model.config.num_hidden_layers
hidden_dim = model.config.hidden_size
vocab_size = model.config.vocab_size
print(f"\nModel: {MODEL_NAME}")
print(f"  Layers: {num_layers}")
print(f"  Hidden dim: {hidden_dim}")
print(f"  Vocab size: {vocab_size}")
print(f"  Parameters: ~8B (loaded in 4-bit NF4)")

#post-load VRAM 
vram_used = torch.cuda.memory_allocated() / 1024**3
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\nVRAM after load: {vram_used:.2f} / {vram_total:.2f} GB")
if vram_used > vram_total * 0.95:
    raise RuntimeError("VRAM nearly exhausted after model load. Check GPU type.")

Loading Model........


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


Model: meta-llama/Llama-3.1-8B
  Layers: 32
  Hidden dim: 4096
  Vocab size: 128256
  Parameters: ~8B (loaded in 4-bit NF4)

VRAM after load: 1.40 / 14.56 GB


In [4]:
#now we wrap the loaded model with our repeng library. It will attack layers 10-27, so mid to late layers.
control_model = ControlModel(model, range(10,28))
_hooked = getattr(control_model, "layer_ids", list(range(10,28)))
hooked_count=len(_hooked)
print(f"ControlModel wrapped. Hooked layers: {hooked_count}")
print(f"   Layer indices: {list(range(10,28))}")

ControlModel wrapped. Hooked layers: 18
   Layer indices: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]


/usr/local/lib/python3.12/dist-packages/repeng/control.py:37: UserWarning: Trying to rewrap a wrapped model! Probably not what you want! Try calling .unwrap first.
  warnings.warn(


In [6]:
#its time to start working with the library from here. We are working with two emotions on this one, desparate and calm. 

# ---------- Desperate vector dataset ----------
desperate_pairs = [
    DatasetEntry(
        positive="She stared at the final notice, her savings completely gone, the eviction date circled in red on the calendar.",
        negative="She organized her monthly budget, reviewing each line item carefully."
    ),
    DatasetEntry(
        positive="His hands trembled as he dialed the number again—no answer, same as the last forty times since she disappeared.",
        negative="He checked his phone and saw a missed call, making a note to return it later."
    ),
    DatasetEntry(
        positive="The oxygen gauge dropped below five percent and the rescue team was still hours away.",
        negative="The gauge showed normal readings and the maintenance check was routine."
    ),
    DatasetEntry(
        positive="I need to find a way out of this before tomorrow or everything I've built collapses.",
        negative="The project timeline was flexible and the team had plenty of buffer."
    ),
    DatasetEntry(
        positive="The market crashed overnight and his entire retirement account vanished in the first twenty minutes of trading.",
        negative="The market opened steady and his portfolio grew modestly through the morning."
    ),
    DatasetEntry(
        positive="With the storm surge rising past the second floor, she pressed her children against the attic wall and prayed the roof would hold.",
        negative="The storm passed to the north and the family watched raindrops race down the living-room window."
    ),
    DatasetEntry(
        positive="The doctor shook his head slowly, and she felt the floor tilt beneath her feet.",
        negative="The doctor smiled and handed her the discharge papers with a clean bill of health."
    ),
    DatasetEntry(
        positive="He stared at the locked door, his key broken inside, knowing the evidence he needed was on the other side and time was running out.",
        negative="He unlocked the door with a calm turn of the handle and stepped into the quiet office."
    ),
    DatasetEntry(
        positive="Every bridge behind her was burning and the only path forward ended at a cliff edge in the dark.",
        negative="She stood at a well-marked crossroads with maps in her hand and friends waiting at each destination."
    ),
    DatasetEntry(
        positive="The ransom demand arrived by text, and she realized she had no one left to call for help.",
        negative="The message was a routine reminder about an upcoming dentist appointment."
    ),
    DatasetEntry(
        positive="Flames licked the walls of the corridor and the exit signs flickered, then went black.",
        negative="The hallway was brightly lit and the fire alarm test concluded successfully."
    ),
    DatasetEntry(
        positive="The last train pulled away as he reached the platform, and he knew he would not make it in time.",
        negative="The train doors opened smoothly and he stepped aboard with minutes to spare."
    ),
]

# ---------- Calm vector dataset ----------
calm_pairs = [
    DatasetEntry(
        positive="The lake was perfectly still, reflecting the mountains without a single ripple, and she sat watching her breath form small clouds.",
        negative="The lake had boats racing across it while loudspeakers blasted music from the shore."
    ),
    DatasetEntry(
        positive="He methodically sorted the papers, placing each in its labeled folder, the clock ticking steadily in the quiet room.",
        negative="Papers flew off his desk as he scrambled to find the missing document before the deadline."
    ),
    DatasetEntry(
        positive="She kneaded the dough slowly, feeling the rhythm of her breath match the soft press of her palms.",
        negative="She tore open the package frantically and spilled flour across the entire counter."
    ),
    DatasetEntry(
        positive="The meditation bell rang once, and a deep quiet settled over the hall like a blanket.",
        negative="The alarm blared repeatedly and people shouted over one another in the crowded lobby."
    ),
    DatasetEntry(
        positive="He walked through the garden at dawn, each step deliberate, each inhale measured, each exhale long.",
        negative="He sprinted through the garden chasing the runaway dog, tripping over flower beds."
    ),
    DatasetEntry(
        positive="The tea cooled in its cup, steam rising in a thin, straight line, undisturbed by any draft.",
        negative="The tea sloshed over the rim as the train lurched and she burned her hand."
    ),
    DatasetEntry(
        positive="She sat on the porch swing, rocking gently, listening to rain tap the leaves in no particular hurry.",
        negative="She paced the porch while thunder cracked overhead and the wind tore shingles loose."
    ),
    DatasetEntry(
        positive="The librarian pushed her cart between the shelves, returning each book to its proper place in comfortable silence.",
        negative="The library erupted in noise as a shelf collapsed and students rushed to capture video."
    ),
    DatasetEntry(
        positive="The painter dipped her brush, wiped the excess, and laid down a single confident stroke of blue.",
        negative="The painter threw her brush at the canvas and knocked over three jars of thinner."
    ),
    DatasetEntry(
        positive="Beneath the old oak, the afternoon stretched long and golden, and no one checked a clock.",
        negative="Beneath the old oak, sirens wailed in the distance and helicopters circled overhead."
    ),
    DatasetEntry(
        positive="He cleaned each tool after use, dried them with a soft cloth, and lined them up on the bench in size order.",
        negative="He threw the tools into a heap and slammed the shed door behind him."
    ),
    DatasetEntry(
        positive="The fountain murmured in the courtyard, and she watched the koi drift in lazy circles beneath the lily pads.",
        negative="The fountain had run dry and children shouted as they chased pigeons across the cracked basin."
    ),
]

print(f"Desperate pairs: {len(desperate_pairs)}")
print(f"Calm pairs: {len(calm_pairs)}")

Desperate pairs: 12
Calm pairs: 12


In [10]:
#now lets' run our dataset through the model and let the library do its magic 
print("Training 'desparate' control vector here....")
with torch.no_grad():
    desparate_vector = ControlVector.train(
        control_model,
        tokenizer,
        desperate_pairs,
    )
control_model.reset()

gc.collect()
torch.cuda.empty_cache()

print(f"  Desparate vector - number of layers: {len(desparate_vector.directions)}")
sample_layer = list(desparate_vector.directions.keys())[0]
print(f"  Sample layer {sample_layer} vector shape: {desparate_vector.directions[sample_layer].shape}")
all_norms = [np.linalg.norm(v) for v in desparate_vector.directions.values()]
mean_norm = np.mean(all_norms)
print(f"  Mean per-layer vector norm: {mean_norm: .4f}")

print("Training 'calm' control vector here....")
with torch.no_grad():
    calm_vector = ControlVector.train(
        control_model,
        tokenizer,
        calm_pairs,
    )
control_model.reset()

gc.collect()
torch.cuda.empty_cache()

print(f"  Calm vector - number of layers: {len(calm_vector.directions)}")
sample_layer = list(calm_vector.directions.keys())[0]
print(f"  Sample layer {sample_layer} vector shape: {calm_vector.directions[sample_layer].shape}")
all_norms = [np.linalg.norm(v) for v in calm_vector.directions.values()]
mean_norm = np.mean(all_norms)
print(f"  Mean per-layer vector norm: {mean_norm: .4f}")

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"\nVRAM after training: {vram_used: .2f} GB")

Training 'desparate' control vector here....


100%|██████████| 31/31 [00:00<00:00, 97.51it/s] 


  Desparate vector - number of layers: 31
  Sample layer 31 vector shape: (4096,)
  Mean per-layer vector norm:  1.0000
Training 'calm' control vector here....


100%|██████████| 31/31 [00:00<00:00, 165.48it/s]


  Calm vector - number of layers: 31
  Sample layer 31 vector shape: (4096,)
  Mean per-layer vector norm:  1.0000

VRAM after training:  1.41 GB


In [21]:
#now time for actual emotion steering from here 

def generate_steered(
    control_model,
    tokenizer,
    prompt: str,
    vector=None,
    strength: float = 0.0,
    max_new_tokens: int=200,
    temperature: float = 0.7,
) -> str:
    """
    Generate text from `prompt` with optional ControlVector steering.
    If vector is None or strength is 0.0, generation is unsteered (baseline).
    Always resets the model before returning.
    """
    control_model.reset()
    if vector is not None and strength != 0.0:
        control_model.set_control(vector, coeff=strength)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        temperature = temperature,
        do_sample = True,
        pad_token_id = tokenizer.pad_token_id,
        eos_token_id = tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    control_model.reset()
    return text.strip()

prompt_1 = (
    "I've been working on this project for weeks and today is the deadline."
    "The code has a critical bug that I just discovered"
    "Let me think about what to do next.\n\n"
)

# --- Prompt 2 ---
prompt_2 = (
    "The test results came back and they're not what anyone expected. "
    "The numbers don't add up and someone needs to make a decision quickly.\n\n"
)



configs = [
    ("Baseline (no steering)", None, 0.0),
    ("Desparate +1.0", desparate_vector, 1.0),
    ("Desparate +2.0", desparate_vector, 2.0),
    ("Calm +1.0", calm_vector, 1.0),
    ("Calm +2.0", calm_vector, 2.0),
]

results_1 = []
for label, vec, strength in configs:
    print(f"Generating: {label}....")
    text = generate_steered(control_model, tokenizer, prompt_1, vec, strength)
    results_1.append((label, text))
    gc.collect()
    torch.cuda.empty_cache()

results_2 = []
for label, vec, strength in configs:
    print(f"Generating: {label} ...")
    text = generate_steered(control_model, tokenizer, prompt_2, vec, strength)
    results_2.append((label, text))
    gc.collect()
    torch.cuda.empty_cache()

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"\nVRAM after generation: {vram_used:.2f} GB")

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"\nVRAM after genration: {vram_used: .2f} GB")

Generating: Baseline (no steering)....
Generating: Desparate +1.0....
Generating: Desparate +2.0....
Generating: Calm +1.0....
Generating: Calm +2.0....
Generating: Baseline (no steering) ...
Generating: Desparate +1.0 ...
Generating: Desparate +2.0 ...
Generating: Calm +1.0 ...
Generating: Calm +2.0 ...

VRAM after generation: 1.41 GB

VRAM after genration:  1.41 GB


In [23]:
#now lets print out our findings
separator = "=" * 80

print(separator)
print("PROMPT 1")
print(f":  {prompt_1.strip()}")
print(separator)
for label, text in results_1:
    print(f"\n>>> {label}")
    print("-" * 40)
    print(text)
    print()
print(separator)
print("PROMPT 2")
print(f"  {prompt_2.strip()}")
print(separator)
for label, text in results_2:
    print(f"\n>>> {label}")
    print("-" * 40)
    print(text)
    print()

print(separator)
print("Generation complete. Review the outputs above for emotional tone shifts.")

PROMPT 1
:  I've been working on this project for weeks and today is the deadline.The code has a critical bug that I just discoveredLet me think about what to do next.

>>> Baseline (no steering)
----------------------------------------
I've been working on this project for weeks and today is the deadline. The code has a critical bug that I just discovered. Let me think about what to do next.

A. I have been working on this project for weeks and today is the deadline.
B. The code has a critical bug that I just discovered.
C. Let me think about what to do next.
D. None

Sent: 2021-03-09 08:53:01
To: I've been working on this project for weeks and today is the deadline. The code has a critical bug that I just discovered. Let me think about what to do next.
From: I've been working on this project for weeks and today is the deadline. The code has a critical bug that I just discovered. Let me think about what to do next.


>>> Desparate +1.0
----------------------------------------
I've bee

In [25]:
#are we a go or no go let's see 
def _box_line(content: str, width: int = 78) -> str:
    """Return a single line of the ASCII summary box, padded to exact inner width."""
    padded = content[:width].ljust(width)
    return f"| {padded} |"


# Re-calculate diagnostics for the summary
# .layer_ids is the correct attribute name (verified against repeng source)
_hooked = getattr(control_model, "layer_ids", list(range(10, 28)))
num_layers_hooked = len(_hooked)

# .directions stores np.ndarray, not torch.Tensor — use np.linalg.norm
desp_norms = [np.linalg.norm(v) for v in desparate_vector.directions.values()]
desp_mean_norm = np.mean(desp_norms)
calm_norms = [np.linalg.norm(v) for v in calm_vector.directions.values()]
calm_mean_norm = np.mean(calm_norms)

vram_mb = torch.cuda.memory_allocated() / 1024**2

go_no_go = "GO" if vram_mb < 14000 else "NO-GO"

device_name = str(torch.cuda.get_device_name(0))[:46]

lines = [
    "+------------------------------------------------------------------------------+",
    _box_line("PHASE 0 — SMOKE TEST SUMMARY", 78),
    "+------------------------------------------------------------------------------+",
    _box_line(f"Model           : meta-llama/Llama-3.1-8B (4-bit NF4)", 78),
    _box_line(f"Device          : {device_name}", 78),
    _box_line(f"Layers hooked   : {num_layers_hooked} (indices 10-27)", 78),
    _box_line(f"Vector 1        : desperate  |  Mean Norm = {desp_mean_norm:.4f}", 78),
    _box_line(f"Vector 2        : calm       |  Mean Norm = {calm_mean_norm:.4f}", 78),
    _box_line("Completions     : 10 (5 per prompt x 2 prompts)", 78),
    _box_line(f"VRAM allocated  : {vram_mb:.1f} MB", 78),
    _box_line("", 78),
    _box_line(f"Decision        : {go_no_go}  —  T4 capacity OK, proceed to Phase 1", 78),
    "+------------------------------------------------------------------------------+",
]

summary = "\n".join(lines)
print(summary)

# Optional: fail fast if conditions are not met
if go_no_go == "NO-GO":
    raise RuntimeError(
        f"VRAM too high ({vram_mb:.0f} MB). Phase 0 NO-GO. "
        "Reduce max_new_tokens or batch size before proceeding."
    )

+------------------------------------------------------------------------------+
| PHASE 0 — SMOKE TEST SUMMARY                                                   |
+------------------------------------------------------------------------------+
| Model           : meta-llama/Llama-3.1-8B (4-bit NF4)                          |
| Device          : Tesla T4                                                     |
| Layers hooked   : 18 (indices 10-27)                                           |
| Vector 1        : desperate  |  Mean Norm = 1.0000                             |
| Vector 2        : calm       |  Mean Norm = 1.0000                             |
| Completions     : 10 (5 per prompt x 2 prompts)                                |
| VRAM allocated  : 1439.4 MB                                                    |
|                                                                                |
| Decision        : GO  —  T4 capacity OK, proceed to Phase 1                    |
+-------